# 完整持仓路径与真实逐笔成本的分配优化

目标保持扣费后稳定跑赢 SOXX 并控制回撤。本轮是一项固定机制实验，结束不等于目标达成。检验自有持仓/回撤反馈与完整连续收益训练，是否使分钟信息获得可实现增量；已有 CPPI、波动率与趋势规则负结果不重新搜索。

三个同结构输入对照：无市场信息、39项日级信息、日级加12项分钟信息。数据从真实分钟覆盖起点2018-10-12开始，严格成熟年度训练，保留全部种子。冻结合同见 plan.json。

In [1]:
import os,sys,json
from pathlib import Path
root=Path.cwd()
while not (root/"SPEC.md").is_file(): root=root.parent
sys.path.insert(0,str(root));os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
from src.workflow.full_path_tiered import prepare_full_path, run_full_path
card=root/"research/full_path_tiered_allocation_20261013"
plan=json.loads((card/"plan.json").read_text())
prepared=prepare_full_path(root=root,card=card,plan=plan)
prepared["audit"]

{'allocator_source': {'recorder_id': '68e31a0a0fad44e7ba1d1f7d856c7a0d',
  'experiment': 'minute_information_pit_allocator_20261012',
  'experiment_id': '905520356740140469',
  'recorder_uri': '/Users/rziiiii/dev/workspace/research/minute_information_pit_allocator_20261012/qlib_recorder',
  'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/minute_information_pit_allocator_20261012/qlib_recorder/905520356740140469/68e31a0a0fad44e7ba1d1f7d856c7a0d',
  'summary_sha256': '61c91f958323cf51b43f4b1554eeca5687011d30771e80bc7ff02d16b6d577ca',
  'verification_manifest_sha256': 'ac90270cf66164e5fb9bca62b50ab86eeab7733bd0146799c45d97937f94e82a',
  'artifacts': {'inputs/features': {'artifact': 'inputs/features',
    'artifact_id': '68e31a0a0fad44e7ba1d1f7d856c7a0d:inputs/features',
    'sha256': '482d6a908ae95230143dc08014732c3247d22ccb1560f8509f6ac564a4d647ec',
    'bytes': 1970939,
    'manifest_entry': 'qlib_recorder/905520356740140469/68e31a0a0fad44e7ba1d1f7d856c7a0d/artifacts/inputs

In [2]:
summary=run_full_path(root=root,card=card,plan=plan,prepared=prepared)
import pandas as pd
from IPython.display import Markdown,display
display(pd.read_csv(card/"backtest_benchmark_comparison.csv"))
display(Markdown((card/"conclusion.md").read_text()))

,fold,family,seed,slippage_bps,sessions,start_signal,last_exit,terminal_wealth,return_pct,cagr_pct,...,up_day_conditional_relative_return_pct,down_day_conditional_relative_return_pct,conditional_returns_are_noncontiguous_statistics,risk_policy,annualization_sessions,maximum_monitored_drawdown_pct,average_cash_pct,orders,broker_and_external_fees_usd,slippage_reserve_usd
0,development_2021,no_market,ensemble,0,252,2021-01-04,2022-01-04,1.386920,38.692037,38.692037,...,-25.807522,27.090180,True,reported against benchmark; no implicit fixed ...,252,-12.705951,17.016978,63,23.574666,0.000000
1,development_2021,daily,ensemble,0,252,2021-01-04,2022-01-04,1.198091,19.809123,19.809123,...,-30.893373,17.866541,True,reported against benchmark; no implicit fixed ...,252,-19.806174,50.956222,426,371.085702,0.000000
2,development_2021,minute,ensemble,0,252,2021-01-04,2022-01-04,1.369095,36.909512,36.909512,...,-15.549925,10.218355,True,reported against benchmark; no implicit fixed ...,252,-15.577813,45.228330,437,501.726037,0.000000
3,development_2021,soxx,ensemble,0,252,2021-01-04,2022-01-04,1.470888,47.088817,47.088817,...,0.000000,0.000000,True,reported against benchmark; no implicit fixed ...,252,-15.019721,0.005050,1,1.756860,0.000000
4,development_2021,no_market,ensemble,2,252,2021-01-04,2022-01-04,1.386786,38.678557,38.678557,...,-25.368595,26.502155,True,reported against benchmark; no implicit fixed ...,252,-12.708315,17.010129,62,23.224322,21.590921
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,reused_outer,minute,172,2,944,2023-01-03,2026-10-09,2.996284,199.628418,34.036608,...,-28.477845,-14.106626,True,reported against benchmark; no implicit fixed ...,252,-55.096442,46.380495,1731,3829.760854,7393.041290
62,reused_outer,minute,273,2,944,2023-01-03,2026-10-09,1.572204,57.220392,12.838649,...,-95.468625,611.370921,True,reported against benchmark; no implicit fixed ...,252,-58.470786,61.449038,1571,3419.414300,6268.342771
63,reused_outer,anchor,frozen,0,944,2023-01-03,2026-10-09,3.289944,228.994402,37.424142,...,-85.444272,363.397905,True,reported against benchmark; no implicit fixed ...,252,-29.501788,48.186722,1281,1558.186235,0.000000
64,reused_outer,anchor,frozen,2,944,2023-01-03,2026-10-09,3.796732,279.673188,42.781845,...,-85.757204,446.552537,True,reported against benchmark; no implicit fixed ...,252,-27.073943,48.309473,1248,1614.361341,2252.524685


# 完整路径扣费优化结论

已执行 27 次严格成熟训练和 66 个 Qlib 原生账户，完整持仓、现金与逐笔费用独立对账。输入从2018真实分钟起点扩充；所有种子保留。

|窗口|模型|滑点bp|净收益%|最大回撤%|相对SOXX财富|
|---|---|---:|---:|---:|---:|
|development_2021|no_market|2|38.68|-10.53|0.9441|
|development_2021|daily|2|18.83|-15.39|0.8089|
|development_2021|minute|2|35.46|-14.11|0.9222|
|development_2021|soxx|2|46.89|-12.44|1.0000|
|development_2022|no_market|2|-34.35|-44.59|1.0251|
|development_2022|daily|2|-49.51|-61.61|0.7883|
|development_2022|minute|2|-39.95|-52.94|0.9377|
|development_2022|soxx|2|-35.96|-47.08|1.0000|
|reused_outer|no_market|2|223.57|-33.25|0.6634|
|reused_outer|daily|2|444.96|-40.90|1.1173|
|reused_outer|minute|2|130.96|-52.28|0.4735|
|reused_outer|soxx|2|387.73|-42.87|1.0000|
|reused_outer|anchor|2|279.67|-24.17|0.7784|

决定：`not_promoted_full_path_gates_failed`。门槛：{"net_excess_every_fold_and_cost": false, "drawdown_30pct_every_fold_and_cost": false, "cvar_no_worse_than_soxx": false, "rolling_win_fraction_at_least_70pct": false, "bootstrap_excess_and_minute_increment": false, "independent_forward_qualified": false}。年度/滚动与区块差异见 sidecars，所有费用和 seed 原生证据保留在 NAS Recorder。

风险：历史复用不构成新留出期，真实首收/成交/股息资格仍缺；整数梯度、训练片段初始现金与软回撤处罚都有限制。模型中先读已知价格/过去状态，再使用未来成交价结算，没有把当前成交价作为当前 NN 输入。仓位目标约束不保证持有漂移后的约束。

目标保持 active；本实验结束不等于稳定超额达成。下一步：qualify frozen independent forward identity if historical gates pass; otherwise close this fixed route and qualify independent release-time semiconductor capacity data。原模拟盘 release 不变。


# 完整路径扣费优化结论

已执行 27 次严格成熟训练和 66 个 Qlib 原生账户，完整持仓、现金与逐笔费用独立对账。输入从2018真实分钟起点扩充；所有种子保留。

|窗口|模型|滑点bp|净收益%|最大回撤%|相对SOXX财富|
|---|---|---:|---:|---:|---:|
|development_2021|no_market|2|38.68|-10.53|0.9441|
|development_2021|daily|2|18.83|-15.39|0.8089|
|development_2021|minute|2|35.46|-14.11|0.9222|
|development_2021|soxx|2|46.89|-12.44|1.0000|
|development_2022|no_market|2|-34.35|-44.59|1.0251|
|development_2022|daily|2|-49.51|-61.61|0.7883|
|development_2022|minute|2|-39.95|-52.94|0.9377|
|development_2022|soxx|2|-35.96|-47.08|1.0000|
|reused_outer|no_market|2|223.57|-33.25|0.6634|
|reused_outer|daily|2|444.96|-40.90|1.1173|
|reused_outer|minute|2|130.96|-52.28|0.4735|
|reused_outer|soxx|2|387.73|-42.87|1.0000|
|reused_outer|anchor|2|279.67|-24.17|0.7784|

决定：`not_promoted_full_path_gates_failed`。门槛：{"net_excess_every_fold_and_cost": false, "drawdown_30pct_every_fold_and_cost": false, "cvar_no_worse_than_soxx": false, "rolling_win_fraction_at_least_70pct": false, "bootstrap_excess_and_minute_increment": false, "independent_forward_qualified": false}。年度/滚动与区块差异见 sidecars，所有费用和 seed 原生证据保留在 NAS Recorder。

风险：历史复用不构成新留出期，真实首收/成交/股息资格仍缺；整数梯度、训练片段初始现金与软回撤处罚都有限制。模型中先读已知价格/过去状态，再使用未来成交价结算，没有把当前成交价作为当前 NN 输入。仓位目标约束不保证持有漂移后的约束。

目标保持 active；本实验结束不等于稳定超额达成。下一步：qualify frozen independent forward identity if historical gates pass; otherwise close this fixed route and qualify independent release-time semiconductor capacity data。原模拟盘 release 不变。

训练/跨期差异：2023冻结的三只 minute 模型训练区块 log 超额均值约0.49–0.51，训练区块最大回撤约21.6–23.1%；复用外窗却相对SOXX财富仅0.474、监测回撤52.7%。日级模型复用外窗相对财富1.117，但开发期两个年份均落后，外窗监测回撤41.4%。这显示全路径目标和成本修正不能制造可泛化的信息；不以外窗收益挑日级模型晋升，不再搜索本固定路线的罚系数、epoch或seed。
